# 🚀 ViSceT5 — **PreSTU SplitOCR + MRA** Pretrain → Finetune · Colab **A100**

Pretrain **PreSTU SplitOCR** (đúng paper): sắp OCR theo thứ tự đọc, cắt ngẫu nhiên → prompt chứa phần đầu, mô hình **sinh phần còn lại từ ảnh** (prefix/target rời nhau, buộc ĐỌC pixel; `full_ocr_prob=0.2` thỉnh thoảng sinh toàn bộ). Không bbox/ground. Kèm **MRA** đúng Feast-Your-Eyes: ViT-L/14 **336** + CNN **1024** (đặc trưng CUỐI của CNN bơm vào **3 stage cuối** ViT), VS TẮT, `VISION_LR_SCALE=0.2`.

Chống overfit (dữ liệu ít): **cosine LR + 6 epoch**. Sau pretrain → finetune ViTextVQA giữ MRA.

Nhánh `exp/mra-pretrain`. Runtime → **A100**. Internet ON.

## 1. Clone Codebase & Checkout Nhánh Pretrain
Tự động phát hiện môi trường (Kaggle hoặc Colab), đồng bộ repository từ GitHub và chuyển sang nhánh `exp/pretrain-gen-all` chứa các cải tiến mới nhất.

In [ ]:
import os
import sys

# Tự động phát hiện thư mục làm việc (Kaggle: /kaggle/working | Colab: /content)
WORK_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "/content"
REPO_DIR = os.path.join(WORK_DIR, "ViSceT5")

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Kussssssss/ViSceT5.git {REPO_DIR}

%cd {REPO_DIR}
!git fetch origin
!git checkout exp/mra-pretrain
!git pull origin exp/mra-pretrain
!git log --oneline -3

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 2. Cài Đặt Môi Trường Chuẩn (Transformers 4.45.2 Cố Định)
Gỡ các phiên bản thư viện mặc định của Kaggle và cài đặt chính xác các phiên bản tương thích từ `requirements.txt`.

In [ ]:
%%capture
!pip uninstall -y transformers peft accelerate 2>/dev/null || true
!pip install -q -r requirements.txt
!pip install -q git+https://github.com/salaniz/pycocoevalcap
!pip install -q --upgrade --no-cache-dir gdown

In [ ]:
# Kiểm tra xác nhận phiên bản môi trường
import torch
import transformers
print(f"✅ PyTorch Version: {torch.__version__} (CUDA Available: {torch.cuda.is_available()})")
print(f"✅ Transformers Version: {transformers.__version__} (Yêu cầu cố định: 4.45.2)")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)} (Count: {torch.cuda.device_count()})")
assert transformers.__version__.startswith("4.45"), f"Cảnh báo: Cần transformers 4.45.x để khớp kiến trúc module, hiện tại là {transformers.__version__}"

## 3. Chuẩn Bị Dữ Liệu Tiền Huấn Luyện (VinText + EVJVQA)
Tự động tải các file nén Image và OCR từ Google Drive theo cấu hình `configs/data/VinText.yaml` và `configs/data/EVJVQA.yaml`, ghép cặp ảnh-OCR và lưu cache vào `output/pretrain`.

In [ ]:
# Định vị thư mục lưu dataset CSV đồng bộ với visualize và trainer
%env OUTPUT_PATH=./output/pretrain
!python scripts/prepare_dataset.py --config configs/data/VinText.yaml,configs/data/EVJVQA.yaml

## 4. Khởi Tạo Trọng Số Mô Hình (ViT5 Base & CLIP ViT)
Khởi tạo `OpenViVQAModel`, tải các trọng số nền tảng ViT5 và CLIP-ViT, kiểm tra tính toàn vẹn số học.

In [ ]:
!python scripts/init_model.py

## 5. Chạy Huấn Luyện PreSTU SplitOCR Pre-Training

### Cấu hình tối ưu toàn diện:
* **Epochs:** 10
* **Batch size:** 4 (per device) $\times$ 4 (gradient accumulation) = Effective Batch Size 16
* **Learning rate:** $1\times 10^{-4}$ (ViT5) và $1\times 10^{-5}$ (CLIP ViT unfrozen 4 layers)
* **Loss balance:** $\lambda_{\text{bbox}} = 0.3$
* **Vision Unfreeze:** Top-4 layers (`vision_unfreeze_last_n = 4`) + post-layernorm
* **Target Split:** Spatial Region Clustering (Khoanh vùng cụm không gian)
* **Output dir:** `/kaggle/working/pretrain_output`

### 5a. SMOKE test PreSTU+MRA (vài step) — bắt lỗi wiring trước khi chạy full

In [ ]:
# SMOKE: vài step, bắt lỗi wiring PreSTU SplitOCR + MRA trước khi chạy full.
!VISION_LR_SCALE=0.2 python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --pretrain_gen_only True \
    --pretrain_split_mode sequential \
    --pretrain_full_ocr_prob 0.2 \
    --vision_unfreeze_last_n 4 \
    --tf32 True \
    --smoke_test True \
    --output_dir ./output/pretrain_mra_smoke \
    --logging_dir ./output/pretrain_mra_smoke/logs

### 5b. FULL PreSTU SplitOCR + MRA pretrain (10 epoch)

In [ ]:
# PreSTU SplitOCR (ĐÚNG paper): sinh phần OCR text còn lại từ ảnh (prefix->target rời nhau).
# split_mode=sequential, full_ocr_prob=0.2 (curriculum độ dài target — hợp dữ liệu ít).
# Chống overfit: cosine LR decay + 6 epoch. MRA: ViT 224 + CNN 1024 (đặc trưng CUỐI -> 3 stage cuối ViT), VS TẮT.
!VISION_LR_SCALE=0.2 python training/pretrain.py configs/pretrain.yaml \
    --dataset_name "VinText,EVJVQA" \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --pretrain_gen_only True \
    --pretrain_split_mode sequential \
    --pretrain_full_ocr_prob 0.2 \
    --vision_unfreeze_last_n 4 \
    --num_train_epochs 6 \
    --lr_scheduler_type cosine \
    --warmup_ratio 0.05 \
    --learning_rate 0.0001 \
    --weight_decay 0.01 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 4 \
    --tf32 True \
    --save_total_limit 2 \
    --output_dir ./output/pretrain_mra \
    --logging_dir ./output/pretrain_mra/logs

## 6. Trực Quan Hóa Kết Quả & Attention Heatmap (Interactive Visual Inspection)

Cell này tải checkpoint tốt nhất vừa huấn luyện và trực quan hóa 3 khung hình song song:
1. **Khung 1 (Ground-Truth):** Ảnh gốc + Hộp BBox Tiền tố (Xanh dương) + Hộp BBox Hậu tố Mục tiêu trong vùng khoanh (Xanh lá) + Chuỗi từ Suffix chuẩn.
2. **Khung 2 (Model Prediction):** Ảnh gốc + Hộp BBox Hậu tố mô hình dự đoán (Đỏ) + Chuỗi từ Suffix do ViT5 Decoder sinh ra qua Beam Search.
3. **Khung 3 (Visual Focus Attention Heatmap):** Bản đồ nhiệt chú ý không gian của Visual Search (AVF) đè lên ảnh gốc, chỉ rõ vùng mắt mô hình đang tập trung nhìn khi sinh từ vựng.

In [ ]:
# ⚠️ Trực quan hóa Attention Heatmap dùng AVF (Visual Search) — MRA chạy VS TẮT nên bỏ qua khung heatmap.
# Nếu muốn xem bbox/суffix dự đoán, chạy visualize với model VS-on riêng. (bỏ qua ở pipeline MRA)
print('Bỏ qua visualize AVF heatmap ở chế độ MRA (VS off).')

## 7. Chuyển Giao Sang Downstream VQA (Transfer Learning to Fine-Tuning)

Sau khi tiền huấn luyện hoàn tất, mô hình đã sẵn sàng chuyển giao tri thức sang bài toán Scene-Text VQA tiếng Việt (**ViTextVQA**).
Toàn bộ trọng số của `vit5.encoder`, `vit5.decoder`, `qa_clip.vision_model` (4 lớp thích ứng) và `visual_search` được nạp nguyên vẹn vào `training/finetune.py`.

In [ ]:
# Finetune downstream ViTextVQA, warm-start tu checkpoint PreSTU+MRA, GIU MRA (VS off, 768).
# fp32 + TF32; batch 4 x accum 2 = effective 8 (khop baseline de cong bang ablation).
!python training/finetune.py configs/finetune.yaml \
    --dataset_name "ViTextVQA" \
    --model_name_or_path ./output/pretrain_mra \
    --ablation_use_mra True \
    --ablation_use_vs False \
    --mra_high_res 1024 \
    --num_train_epochs 5 \
    --per_device_train_batch_size 4 \
    --gradient_accumulation_steps 2 \
    --learning_rate 0.00003 \
    --tf32 True \
    --output_dir ./output/finetune_mra \
    --logging_dir ./output/finetune_mra/logs

## 8. Nén Checkpoint Để Tải Về & Tùy Chọn Upload HuggingFace

In [ ]:
# Nén toàn bộ checkpoint pretrain và ảnh visualizations thành file ZIP để tải về từ giao diện Kaggle
!zip -r /kaggle/working/ViSceT5_PreSTU_Pretrain.zip /kaggle/working/pretrain_output
print("✅ Đã nén thành công checkpoint tại /kaggle/working/ViSceT5_PreSTU_Pretrain.zip")

In [ ]:
# (Tùy chọn) Đăng tải trực tiếp checkpoint lên HuggingFace Hub nếu có Token
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path="/kaggle/working/pretrain_output",
#     repo_id="your-username/ViSceT5-PreSTU-Pretrained",
#     repo_type="model",
#     token="your_hf_token_here"
# )